In [15]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from proyecto_mlops_equipo_9.config import (PROCESSED_DATA_DIR, CLEAN_DATA_DIR)

In [36]:
input_path: Path = PROCESSED_DATA_DIR / "datos_feature_engineered.parquet"
output_path: Path = PROCESSED_DATA_DIR / "partitions"

TIMESTAMP_COL = "inicio_intervalo"

In [ ]:
# Chronological partition boundaries default
SPLITS = {
    "train": {
        "start": "2025-01-01",
        "end": "2026-04-01"
    },
    "validation": {
        "start": "2026-04-01",
        "end": "2026-06-01"
    },
    "drift": {
        "start": "2026-06-01",
        "end": "2026-10-01"
    }
}

In [17]:
df = pd.read_parquet(input_path)
df

,inicio_intervalo,area_id,salidas,duracion_prom_salidas,velocidad_prom_salidas,proporcion_viajes_cortos_salidas,proporcion_cortos_distinta_estacion_salidas,llegadas,duracion_prom_llegadas,velocidad_prom_llegadas,...,dia_anio,anio,es_fin_semana,es_hora_pico,hora_sin,hora_cos,dia_semana_sin,dia_semana_cos,salidas_siguiente_hora,llegadas_siguiente_hora
0,2025-01-01 06:00:00,2,0,NaN,NaN,NaN,NaN,1,17.116667,7.105170,...,1,2025,0,0,1.000000,6.123234e-17,0.974928,-0.222521,0.0,0.0
1,2025-01-01 06:15:00,2,0,NaN,NaN,NaN,NaN,1,12.116667,10.713110,...,1,2025,0,0,0.997859,-6.540313e-02,0.974928,-0.222521,0.0,0.0
2,2025-01-01 08:30:00,2,1,3.116667,5.331619,0.0,0.0,0,NaN,NaN,...,1,2025,0,1,0.793353,-6.087614e-01,0.974928,-0.222521,0.0,0.0
3,2025-01-01 09:45:00,2,1,5.300000,8.122352,0.0,0.0,0,NaN,NaN,...,1,2025,0,1,0.555570,-8.314696e-01,0.974928,-0.222521,2.0,2.0
4,2025-01-01 10:30:00,2,0,NaN,NaN,NaN,NaN,1,11.800000,6.258241,...,1,2025,0,0,0.382683,-9.238795e-01,0.974928,-0.222521,3.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7328183,2026-09-30 17:15:00,519,0,NaN,NaN,NaN,NaN,1,13.000000,12.266525,...,273,2026,0,1,-0.980785,-1.950903e-01,0.974928,-0.222521,0.0,4.0
7328184,2026-09-30 18:15:00,519,0,NaN,NaN,NaN,NaN,1,1.700000,8.712828,...,273,2026,0,1,-0.997859,6.540313e-02,0.974928,-0.222521,1.0,4.0
7328185,2026-09-30 19:00:00,519,1,4.783333,6.266805,0.0,0.0,1,10.483333,10.442725,...,273,2026,0,1,-0.965926,2.588190e-01,0.974928,-0.222521,2.0,4.0
7328186,2026-09-30 19:15:00,519,1,7.083333,15.455233,0.0,0.0,1,2.766667,5.353665,...,273,2026,0,1,-0.946930,3.214395e-01,0.974928,-0.222521,2.0,4.0


In [11]:
df.dtypes

inicio_intervalo                                datetime64[us]
area_id                                                  int64
salidas                                                  int64
duracion_prom_salidas                                  float64
velocidad_prom_salidas                                 float64
proporcion_viajes_cortos_salidas                       float64
proporcion_cortos_distinta_estacion_salidas            float64
llegadas                                                 int64
duracion_prom_llegadas                                 float64
velocidad_prom_llegadas                                float64
proporcion_viajes_cortos_llegadas                      float64
proporcion_cortos_distinta_estacion_llegadas           float64
hora                                                     int32
dia_semana                                               int32
dia_mes                                                  int32
mes                                                    

In [12]:
if TIMESTAMP_COL not in df.columns:
    raise ValueError(
        f"Timestamp column '{TIMESTAMP_COL}' not found."
    )

In [18]:
df[TIMESTAMP_COL] = pd.to_datetime(
    df[TIMESTAMP_COL],
    errors="raise"
)

In [19]:
df = df.sort_values(TIMESTAMP_COL).reset_index(drop=True)
df

,inicio_intervalo,area_id,salidas,duracion_prom_salidas,velocidad_prom_salidas,proporcion_viajes_cortos_salidas,proporcion_cortos_distinta_estacion_salidas,llegadas,duracion_prom_llegadas,velocidad_prom_llegadas,...,dia_anio,anio,es_fin_semana,es_hora_pico,hora_sin,hora_cos,dia_semana_sin,dia_semana_cos,salidas_siguiente_hora,llegadas_siguiente_hora
0,2025-01-01,190,1,9.916667,14.062676,0.0,0.0,0,NaN,NaN,...,1,2025,0,0,0.0,1.0,0.974928,-0.222521,0.0,0.0
1,2025-01-01,35,2,10.433333,3.103320,0.0,0.0,0,NaN,NaN,...,1,2025,0,0,0.0,1.0,0.974928,-0.222521,0.0,0.0
2,2025-01-01,395,1,7.116667,5.560967,0.0,0.0,1,10.750000,7.556001,...,1,2025,0,0,0.0,1.0,0.974928,-0.222521,0.0,0.0
3,2025-01-01,156,1,13.350000,0.000000,0.0,0.0,0,NaN,NaN,...,1,2025,0,0,0.0,1.0,0.974928,-0.222521,0.0,0.0
4,2025-01-01,211,1,10.750000,7.556001,0.0,0.0,0,NaN,NaN,...,1,2025,0,0,0.0,1.0,0.974928,-0.222521,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7328183,2026-10-01,160,0,NaN,NaN,NaN,NaN,1,9.033333,15.477127,...,274,2026,0,0,0.0,1.0,0.433884,-0.900969,0.0,0.0
7328184,2026-10-01,269,0,NaN,NaN,NaN,NaN,1,20.600000,9.448274,...,274,2026,0,0,0.0,1.0,0.433884,-0.900969,0.0,0.0
7328185,2026-10-01,73,0,NaN,NaN,NaN,NaN,1,18.233333,0.000000,...,274,2026,0,0,0.0,1.0,0.433884,-0.900969,0.0,0.0
7328186,2026-10-01,428,0,NaN,NaN,NaN,NaN,1,22.400000,7.212091,...,274,2026,0,0,0.0,1.0,0.433884,-0.900969,0.0,0.0


In [20]:
partitions = {}

for name, dates in SPLITS.items():

    start = pd.Timestamp(dates["start"])
    end = pd.Timestamp(dates["end"])

    mask = (
        (df[TIMESTAMP_COL] >= start) &
        (df[TIMESTAMP_COL] < end)
    )

    partition = df.loc[mask].copy()

    if partition.empty:
        raise ValueError(f"{name} partition is empty.")

    partitions[name] = partition

In [43]:
partitions['train'].head(2)

,inicio_intervalo,area_id,salidas,duracion_prom_salidas,velocidad_prom_salidas,proporcion_viajes_cortos_salidas,proporcion_cortos_distinta_estacion_salidas,llegadas,duracion_prom_llegadas,velocidad_prom_llegadas,...,dia_anio,anio,es_fin_semana,es_hora_pico,hora_sin,hora_cos,dia_semana_sin,dia_semana_cos,salidas_siguiente_hora,llegadas_siguiente_hora
0,2025-01-01,190,1,9.916667,14.062676,0.0,0.0,0,NaN,NaN,...,1,2025,0,0,0.0,1.0,0.974928,-0.222521,0.0,0.0
1,2025-01-01,35,2,10.433333,3.103320,0.0,0.0,0,NaN,NaN,...,1,2025,0,0,0.0,1.0,0.974928,-0.222521,0.0,0.0


In [44]:
partitions['validation'].head(2)

,inicio_intervalo,area_id,salidas,duracion_prom_salidas,velocidad_prom_salidas,proporcion_viajes_cortos_salidas,proporcion_cortos_distinta_estacion_salidas,llegadas,duracion_prom_llegadas,velocidad_prom_llegadas,...,dia_anio,anio,es_fin_semana,es_hora_pico,hora_sin,hora_cos,dia_semana_sin,dia_semana_cos,salidas_siguiente_hora,llegadas_siguiente_hora
5188195,2026-04-01,200,1,5.983333,15.446973,0.0,0.0,0,NaN,NaN,...,91,2026,0,0,0.0,1.0,0.974928,-0.222521,2.0,3.0
5188196,2026-04-01,271,0,NaN,NaN,NaN,NaN,1,10.116667,9.436713,...,91,2026,0,0,0.0,1.0,0.974928,-0.222521,1.0,4.0


In [45]:
partitions['drift'].head(2)

,inicio_intervalo,area_id,salidas,duracion_prom_salidas,velocidad_prom_salidas,proporcion_viajes_cortos_salidas,proporcion_cortos_distinta_estacion_salidas,llegadas,duracion_prom_llegadas,velocidad_prom_llegadas,...,dia_anio,anio,es_fin_semana,es_hora_pico,hora_sin,hora_cos,dia_semana_sin,dia_semana_cos,salidas_siguiente_hora,llegadas_siguiente_hora
5839314,2026-06-01,319,0,NaN,NaN,NaN,NaN,1,23.700000,13.837875,...,152,2026,0,0,0.0,1.0,0.0,1.0,4.0,1.0
5839315,2026-06-01,30,0,NaN,NaN,NaN,NaN,1,3.116667,10.159037,...,152,2026,0,0,0.0,1.0,0.0,1.0,3.0,2.0


In [39]:
output_path.mkdir(parents=True, exist_ok=True)

for name, partition in partitions.items():
    print(name)
    output_file = output_path / f"{name}.parquet"
    print(output_file)

    partition.to_parquet(
            output_file,
            index=False
        )

train
C:\Users\paqui\OneDrive\Documentos\Maestria\MLOPS_Equipo\proyecto_mlops_equipo_9\data\processed\partitions\train.parquet
validation
C:\Users\paqui\OneDrive\Documentos\Maestria\MLOPS_Equipo\proyecto_mlops_equipo_9\data\processed\partitions\validation.parquet
drift
C:\Users\paqui\OneDrive\Documentos\Maestria\MLOPS_Equipo\proyecto_mlops_equipo_9\data\processed\partitions\drift.parquet


# Verificar salida del script

In [42]:
output_df = pd.read_parquet(output_path / 'train.parquet')
output_df.head(2)

,inicio_intervalo,area_id,salidas,duracion_prom_salidas,velocidad_prom_salidas,proporcion_viajes_cortos_salidas,proporcion_cortos_distinta_estacion_salidas,llegadas,duracion_prom_llegadas,velocidad_prom_llegadas,...,dia_anio,anio,es_fin_semana,es_hora_pico,hora_sin,hora_cos,dia_semana_sin,dia_semana_cos,salidas_siguiente_hora,llegadas_siguiente_hora
0,2025-01-01,190,1,9.916667,14.062676,0.0,0.0,0,NaN,NaN,...,1,2025,0,0,0.0,1.0,0.974928,-0.222521,0.0,0.0
1,2025-01-01,35,2,10.433333,3.103320,0.0,0.0,0,NaN,NaN,...,1,2025,0,0,0.0,1.0,0.974928,-0.222521,0.0,0.0


In [46]:
output_df = pd.read_parquet(output_path / 'validation.parquet')
output_df.head(2)

,inicio_intervalo,area_id,salidas,duracion_prom_salidas,velocidad_prom_salidas,proporcion_viajes_cortos_salidas,proporcion_cortos_distinta_estacion_salidas,llegadas,duracion_prom_llegadas,velocidad_prom_llegadas,...,dia_anio,anio,es_fin_semana,es_hora_pico,hora_sin,hora_cos,dia_semana_sin,dia_semana_cos,salidas_siguiente_hora,llegadas_siguiente_hora
0,2026-04-01,200,1,5.983333,15.446973,0.0,0.0,0,NaN,NaN,...,91,2026,0,0,0.0,1.0,0.974928,-0.222521,2.0,3.0
1,2026-04-01,271,0,NaN,NaN,NaN,NaN,1,10.116667,9.436713,...,91,2026,0,0,0.0,1.0,0.974928,-0.222521,1.0,4.0


In [47]:
output_df = pd.read_parquet(output_path / 'drift.parquet')
output_df.head(2)

,inicio_intervalo,area_id,salidas,duracion_prom_salidas,velocidad_prom_salidas,proporcion_viajes_cortos_salidas,proporcion_cortos_distinta_estacion_salidas,llegadas,duracion_prom_llegadas,velocidad_prom_llegadas,...,dia_anio,anio,es_fin_semana,es_hora_pico,hora_sin,hora_cos,dia_semana_sin,dia_semana_cos,salidas_siguiente_hora,llegadas_siguiente_hora
0,2026-06-01,319,0,NaN,NaN,NaN,NaN,1,23.700000,13.837875,...,152,2026,0,0,0.0,1.0,0.0,1.0,4.0,1.0
1,2026-06-01,30,0,NaN,NaN,NaN,NaN,1,3.116667,10.159037,...,152,2026,0,0,0.0,1.0,0.0,1.0,3.0,2.0
